<img src="images/duckdb-logo.png" width="15%" align="right">

# DuckDB

---

Website: https://duckdb.org

In [1]:
import duckdb as duck
import pandas as pd

In [3]:
import duckdb as duck
import io
import zipfile
import requests
import os

url = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_1.zip"

r = requests.get(url)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
    z.extract(csv_file, "/content")

source = os.path.join("/content", csv_file)

df = duck.read_csv(source)

print("Đọc dữ liệu thành công")

Đọc dữ liệu thành công


In [4]:
duck.sql("SELECT * FROM df")

┌───────┬─────────┬───────┬────────────┬───────────┬────────────┬───────────────────┬──────────────────────────┬─────────────────────────────┬─────────────┬─────────────────────────────────┬─────────────────┬────────────────────┬────────────────────┬─────────┬────────────────────────────────────┬─────────────┬─────────────────┬─────────────────┬───────────┬───────────────┬──────────────────┬──────────────────┬─────────┬────────────────────────────────────┬───────────┬───────────────┬───────────────┬─────────┬────────────┬─────────┬──────────┬─────────────────┬──────────┬──────────────────────┬────────────┬─────────┬───────────┬──────────┬────────┬────────────┬─────────┬──────────┬─────────────────┬──────────┬────────────────────┬────────────┬───────────┬──────────────────┬──────────┬────────────────┬───────────────────┬─────────┬─────────┬──────────┬───────────────┬──────────────┬──────────────┬──────────┬───────────────┬───────────────────┬──────────────┬───────────────┬──────────────

### Mean arrival delay

You can now use SQL queries:

In [5]:
duck.sql("""
    SELECT * FROM read_csv(
        '/content/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1.csv',
        strict_mode=false
    )
""")

┌───────┬─────────┬───────┬────────────┬───────────┬────────────┬───────────────────┬──────────────────────────┬─────────────────────────────┬─────────────┬─────────────────────────────────┬─────────────────┬────────────────────┬────────────────────┬─────────┬────────────────────────────────────┬─────────────┬─────────────────┬─────────────────┬───────────┬───────────────┬──────────────────┬──────────────────┬─────────┬────────────────────────────────────┬───────────┬───────────────┬───────────────┬─────────┬────────────┬─────────┬──────────┬─────────────────┬──────────┬──────────────────────┬────────────┬─────────┬───────────┬──────────┬────────┬────────────┬─────────┬──────────┬─────────────────┬──────────┬────────────────────┬────────────┬───────────┬──────────────────┬──────────┬────────────────┬───────────────────┬─────────┬─────────┬──────────┬───────────────┬──────────────┬──────────────┬──────────┬───────────────┬───────────────────┬──────────────┬───────────────┬──────────────

In [6]:
df.columns

['Year',
 'Quarter',
 'Month',
 'DayofMonth',
 'DayOfWeek',
 'FlightDate',
 'Reporting_Airline',
 'DOT_ID_Reporting_Airline',
 'IATA_CODE_Reporting_Airline',
 'Tail_Number',
 'Flight_Number_Reporting_Airline',
 'OriginAirportID',
 'OriginAirportSeqID',
 'OriginCityMarketID',
 'Origin',
 'OriginCityName',
 'OriginState',
 'OriginStateFips',
 'OriginStateName',
 'OriginWac',
 'DestAirportID',
 'DestAirportSeqID',
 'DestCityMarketID',
 'Dest',
 'DestCityName',
 'DestState',
 'DestStateFips',
 'DestStateName',
 'DestWac',
 'CRSDepTime',
 'DepTime',
 'DepDelay',
 'DepDelayMinutes',
 'DepDel15',
 'DepartureDelayGroups',
 'DepTimeBlk',
 'TaxiOut',
 'WheelsOff',
 'WheelsOn',
 'TaxiIn',
 'CRSArrTime',
 'ArrTime',
 'ArrDelay',
 'ArrDelayMinutes',
 'ArrDel15',
 'ArrivalDelayGroups',
 'ArrTimeBlk',
 'Cancelled',
 'CancellationCode',
 'Diverted',
 'CRSElapsedTime',
 'ActualElapsedTime',
 'AirTime',
 'Flights',
 'Distance',
 'DistanceGroup',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'SecurityD

In [7]:
duck.sql("SELECT AVG(ArrDelay) AS avg_delay FROM df")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│     avg_delay      │
│       double       │
├────────────────────┤
│ 3.9810854985512254 │
└────────────────────┘

In [8]:
duck.sql("""
    SELECT Reporting_Airline, ArrDelay
    FROM df
    WHERE Reporting_Airline = 'DL'
    LIMIT 10
""")

┌───────────────────┬──────────┐
│ Reporting_Airline │ ArrDelay │
│      varchar      │  double  │
├───────────────────┼──────────┤
│ DL                │     NULL │
│ DL                │     -1.0 │
│ DL                │    -12.0 │
│ DL                │    -15.0 │
│ DL                │    -11.0 │
│ DL                │    -23.0 │
│ DL                │    -12.0 │
│ DL                │      2.0 │
│ DL                │     -2.0 │
│ DL                │    -11.0 │
├───────────────────┴──────────┤
│ 10 rows            2 columns │
└──────────────────────────────┘

---

## Next →

[Let's wrap up](12-conclusion.ipynb)!